# Zadanie 3: własny agent kodujący

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/03-agenci/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-03`, w załączniku ten notebook **z wynikami komórek**, bez kluczy API.

**Termin:** przed zajęciami 4.

Agent kodujący to harness wokół modelu: pętla, narzędzia do plików i poleceń, historia. W tym zadaniu powstaje najprostsza wersja takiego harnessu w LangGraph, a agent dostaje jedno polecenie: zbudować małą aplikację z testami.

Części:
1. Narzędzia do pracy z plikami i uruchamiania poleceń.
2. Agent jako graf w LangGraph.
3. Uruchomienie agenta na podanym poleceniu i sprawdzenie wyniku.

Kod z [notebooka z zajęć](notebook.ipynb) można kopiować i modyfikować.

> **Uwaga.** Model sam układa polecenia, które wykona komputer. Najbezpieczniej uruchamiać notebook w Colabie albo Codespaces. Na własnym komputerze wszystkie pliki muszą powstawać w katalogu `workspace`, a polecenia wymagają zatwierdzenia przez człowieka albo sprawdzenia, czy nie wychodzą poza ten katalog.

## 0. Konfiguracja

In [ ]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai python-dotenv langchain langchain-openai langgraph fastapi httpx pytest

Dostawcę wybiera się zmienną `LLM_PROVIDER`, tak jak w notebooku z zajęć ([00-dostep-do-llm](../00-dostep-do-llm/README.md)). Zadanie nie wymaga embeddingów. Agent wysyła dużo wywołań, więc dobrze sprawdza się mały i tani model z tool callingiem. Małe modele lokalne często mylą argumenty narzędzi.

In [ ]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "DEEPSEEK_API_KEY"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

MODELS = {
    "openai":   {"chat": "gpt-5-nano", "agent": "gpt-5.4-nano"},
    "azure":    {"chat": "gpt-4o-mini", "agent": "gpt-4o-mini"},
    "deepseek": {"chat": "deepseek-chat", "agent": "deepseek-chat"},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "agent": "gemma4:e4b"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "agent": "copilot/gpt-5-mini"},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")

client = make_client(PROVIDER)
CHAT, AGENT = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["agent"]  # AGENT: model do pętli agenta
print(f"czat: {PROVIDER} ({CHAT}, agent: {AGENT})")

from langchain_openai import ChatOpenAI, AzureChatOpenAI

def make_llm(model):
    if PROVIDER == "azure":
        return AzureChatOpenAI(azure_deployment=model, api_version="2024-12-01-preview")
    return ChatOpenAI(model=model, base_url=str(client.base_url), api_key=client.api_key)

llm = make_llm(AGENT)

## Część 1. Narzędzia

Narzędzia to funkcje z dekoratorem `@tool` z LangChain. Agent potrzebuje co najmniej:

| Narzędzie | Co robi |
|---|---|
| `create_folder(path)` | tworzy katalog |
| `delete_folder(path)` | usuwa katalog razem z zawartością |
| `create_file(path, content)` | tworzy plik z podaną treścią |
| `read_file(path)` | zwraca treść pliku |
| `replace_in_file(path, old, new)` | zamienia fragment pliku; pusty `new` usuwa fragment |
| `list_files(path)` | wypisuje pliki w katalogu |
| `run_command(command)` | uruchamia polecenie powłoki (bash albo PowerShell, zależnie od systemu) i zwraca kod wyjścia, stdout i stderr |

Wymagania:
1. Wszystkie ścieżki przechodzą przez `safe_path` z komórki poniżej.
2. `run_command` działa w katalogu `WORKSPACE` i ma limit czasu (np. 60 s).
3. Błąd (brak pliku, nieudane polecenie) wraca do modelu jako tekst, a nie przerywa programu: model może go wtedy naprawić.
4. Wyniki są przycinane (np. do 4000 znaków), żeby nie zapełniały kontekstu.
5. Docstring każdego narzędzia jasno opisuje, kiedy go użyć: to część promptu.

In [ ]:
from pathlib import Path
from langchain.tools import tool

WORKSPACE = Path("workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

def safe_path(path: str) -> Path:
    """Zamienia ścieżkę względną na bezwzględną i odrzuca wszystko spoza WORKSPACE."""
    full = (WORKSPACE / path).resolve()
    if not full.is_relative_to(WORKSPACE):
        raise ValueError(f"ścieżka poza katalogiem roboczym: {path}")
    return full

# TODO: narzędzia create_folder, delete_folder, create_file, read_file, replace_in_file, list_files, run_command

TOOLS = [
    # TODO
]

In [ ]:
# TODO: krótki test każdego narzędzia bez modelu, w tym próba wyjścia poza WORKSPACE (np. "../plik.txt")

## Część 2. Agent w LangGraph

Graf budowany ręcznie przez `StateGraph` (bez `create_agent`), jak w sekcji 5 notebooka z zajęć:

1. stan `MessagesState`, węzły `model` i `tools`, krawędź warunkowa `tools_condition`,
2. system prompt opisujący pracę agenta kodującego: najpierw plan, potem pliki, potem uruchomienie testów i poprawki, aż testy przejdą,
3. rysunek grafu w notebooku (`draw_mermaid_png`).

In [ ]:
# TODO: SYSTEM, call_model, StateGraph, compile

In [ ]:
# TODO: rysunek grafu

## Część 3. Uruchomienie

Polecenie dla agenta jest podane poniżej. Można zmienić temat aplikacji (np. biblioteka zamiast sklepu), ale zakres musi zostać taki sam: jeden plik aplikacji, SQLite, kilka endpointów i testy.

1. Uruchomienie przez `graph.stream(..., stream_mode="updates", config={"recursion_limit": 60})` z wypisaniem każdego kroku: nazwa narzędzia, skrócone argumenty, skrócony wynik.
2. Na końcu: liczba wywołań modelu, liczba wywołań narzędzi i suma tokenów wejściowych (`usage_metadata` w odpowiedziach modelu).
3. Sprawdzenie wyniku **niezależnie od agenta**: lista plików, treść `app.py` i `test_app.py`, uruchomienie `pytest` z komórki notebooka.

In [ ]:
TASK = """Utwórz małą aplikację FastAPI: sklep internetowy.

- Cały kod aplikacji w jednym pliku app.py, baza SQLite w pliku shop.db.
- Przy starcie aplikacja tworzy tabelę products (id, name, price, stock), a gdy tabela jest pusta, dodaje 5 przykładowych produktów.
- Endpointy:
  - GET /products: lista produktów,
  - GET /products/{id}: jeden produkt, 404 gdy nie istnieje,
  - POST /products: dodanie produktu, zwraca 201,
  - POST /products/{id}/buy: zmniejsza stock o 1, 404 gdy produkt nie istnieje, 409 gdy stock wynosi 0.
- Testy w pliku test_app.py (pytest i TestClient z FastAPI) sprawdzają każdy endpoint, także błędy 404 i 409.
- Uruchom testy poleceniem `python -m pytest -q`. Jeśli nie przechodzą, popraw kod i uruchom je ponownie."""

# TODO: uruchomienie agenta i wypisanie kroków

In [ ]:
# TODO: liczba wywołań modelu i narzędzi, suma tokenów wejściowych

In [ ]:
# TODO: lista plików w WORKSPACE, treść app.py i test_app.py

In [ ]:
# TODO: uruchomienie pytest w WORKSPACE z tej komórki (subprocess), bez udziału agenta